<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# Notebook 03: Foundation Model Training

We train a **decoder foundation model** on the tokenized transaction sequences from notebook 02 using **causal language modeling** (next-token prediction). At each position, the model predicts the next token from all preceding tokens, learning transaction patterns without any fraud labels.

```
Input:  <bos>  AMT_3  MERCH_1498  CAT_RETAIL  MCC_5912  HOUR_05  DOW_0  ...
Target:        AMT_3  MERCH_1498  CAT_RETAIL  MCC_5912  HOUR_05  DOW_0  MONTH_05  ...
```

Every token position provides a training signal, so through millions of predictions the model builds rich representations of spending patterns, merchant relationships, and temporal behaviors.

## Model Configuration

| Parameter | Value | Notes |
|-----------|-------|-------|
| **Architecture** | Decoder-only Transformer | Causal (autoregressive) attention |
| **Hidden Size** | 512 | Transformer hidden dimension |
| **Layers** | 8 | Transformer depth |
| **Attention Heads** | 8 query / 2 KV | Grouped-Query Attention (GQA) |
| **FFN Size** | 1408 | SwiGLU activation |
| **Context Window** | 8192 | RoPE positional encoding |
| **Parameters** | **\~29M** | Compact embedding table |
| **Vocab Size** | \~6,252 | Matches `FinancialTokenizerPipeline` (merchant_hash_size=2,000) |
| **Training Data** | \~263M tokens | \~64K sequences from 19.5M transactions |

### Aligned Tokenizer and Model Vocabulary
The model uses ```merchant_hash_size=2,000```, matching the corpus generated in notebook 02 by ```FinancialTokenizerPipeline```. This eliminates the ~24M dead embedding parameters that would result from a 50K merchant vocabulary trained on a 2K corpus. 

* **Context window vs. corpus length**: the model supports 8192-token sequences, but the corpus generated in notebook 02 uses 4096-token chunks (~315 transactions). This leaves headroom for future experiments with longer transaction histories without retraining the positional encodings. 

## Training Pipeline

Training uses `scripts/train_decoder_model.py` with **NeMo AutoModel** and FSDP2 for distributed training. Configuration is defined in `configs/pretrain_financial_decoder.yaml`; custom tokenizer and dataset are integrated via the `_target_` syntax. Scaling from 1 GPU to multi-node requires only a `torchrun` wrapper -- no code changes.

## Custom Components

We only provide two domain-specific components -- everything else (FSDP2, optimizer, checkpointing, mixed precision) is handled by NeMo AutoModel:

1. **`FinancialTabularTokenizer`** -- custom tokenization for transaction data
2. **`FinancialCLMDataset`** -- next-token prediction data loading (referenced in YAML via `_target_`)

Checkpoints are saved in HuggingFace format (`save_consolidated: true`), enabling downstream use with `AutoModelForCausalLM.from_pretrained()`.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, module="liger_kernel")
warnings.filterwarnings("ignore", message="The '.*' attribute with value.*was provided")

import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
CORPUS_DIR = DATA_DIR / "decoder_corpus"
CORPUS_PATH = CORPUS_DIR / "train_corpus.txt"
VAL_CORPUS_PATH = CORPUS_DIR / "val_corpus.txt"

YAML_CONFIG = PROJECT_ROOT / "configs/pretrain_financial_decoder.yaml"
TRAIN_SCRIPT = PROJECT_ROOT / "scripts/train_decoder_hf.py"

PRETRAINED_MODEL = PROJECT_ROOT / "models/decoder-foundation-model"

# Verify prerequisites from notebook 02
assert CORPUS_PATH.exists(), f"Training corpus not found: {CORPUS_PATH}  (run notebook 02 first)"
assert VAL_CORPUS_PATH.exists(), f"Validation corpus not found: {VAL_CORPUS_PATH}  (run notebook 02 first)"
assert YAML_CONFIG.exists(), f"YAML config not found: {YAML_CONFIG}"
assert TRAIN_SCRIPT.exists(), f"Training script not found: {TRAIN_SCRIPT}"

# Pre-trained checkpoint (used by notebook 04 for inference)
if PRETRAINED_MODEL.exists() and (PRETRAINED_MODEL / "config.json").exists():
    print(f"Pre-trained decoder checkpoint available for notebook 04:")
    print(f"  {PRETRAINED_MODEL}\n")

print(f"Training corpus : {CORPUS_PATH}")
print(f"Validation      : {VAL_CORPUS_PATH} ({'exists' if VAL_CORPUS_PATH.exists() else 'NOT FOUND'})")
print(f"YAML config     : {YAML_CONFIG}")
print(f"Pre-trained     : {PRETRAINED_MODEL}")

Pre-trained decoder checkpoint available for notebook 04:
  /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/models/decoder-foundation-model

Training corpus : /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/data/decoder_corpus/train_corpus.txt
Validation      : /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/data/decoder_corpus/val_corpus.txt (exists)
YAML config     : /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/configs/pretrain_financial_decoder.yaml
Pre-trained     : /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/models/decoder-foundation-model


## 1. Launch Training

Training is configured through `configs/pretrain_financial_decoder.yaml` (model architecture, AdamW + cosine annealing optimizer, FSDP2 distribution, consolidated HF checkpoints). The cell below runs a **30-step demo** on 1 GPU (\~2 min). The pre-trained model used in notebook 04 was trained for \~3,000 steps on 8x A100.

The random baseline loss is `ln(~6,252) ~ 8.7`; loss should drop to \~5–6 within 30 steps.

### YAML Configuration

In [2]:
print(f"=== {YAML_CONFIG} ===\n")
with open(YAML_CONFIG) as f:
    print(f.read())

=== /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/configs/pretrain_financial_decoder.yaml ===

# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Financial Foundation Model Pretraining — Decoder-Only
#
# NeMo AutoModel configuration for pretraining a decoder foundation model on
# financial transaction sequences using causal language modeling

In [3]:
## Check MPS backend
import sys, torch
print(sys.executable)
print(torch.__version__)
print(torch.backends.mps.is_available())
print(torch.ones(1, device="mps"))

/opt/anaconda3/envs/tfm/bin/python3.12
2.11.0
True
tensor([1.], device='mps:0')


### Try smoke run above first!!! 

In [4]:
import os
import sys
import subprocess
from pathlib import Path

SMOKE_DIR = PROJECT_ROOT / "data" / "decoder_corpus" / "smoke"
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

SMOKE_TRAIN = SMOKE_DIR / "train_512.txt"
with open(CORPUS_PATH, "r") as src, open(SMOKE_TRAIN, "w") as dst:
  for i, line in enumerate(src):
      if i >= 512:
          break
      dst.write(line)

DEMO_CHECKPOINT_DIR = PROJECT_ROOT / "models" / "decoder-demo" / "checkpoints"
DEMO_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"

cmd = [
  sys.executable, str(TRAIN_SCRIPT),
  "--train_data", str(SMOKE_TRAIN),
  "--output_dir", str(DEMO_CHECKPOINT_DIR),
  "--max_steps", "3",
  "--seq_length", "512",
  "--per_device_train_batch_size", "1",
  "--save_steps", "1000",
  "--logging_steps", "1",
]

print(" ".join(cmd))
result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=env)
print("returncode:", result.returncode)

/opt/anaconda3/envs/tfm/bin/python3.12 /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/scripts/train_decoder_hf.py --train_data /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/data/decoder_corpus/smoke/train_512.txt --output_dir /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/models/decoder-demo/checkpoints --max_steps 3 --seq_length 512 --per_device_train_batch_size 1 --save_steps 1000 --logging_steps 1

Financial Foundation Model — Decoder-Only Pretraining (HF)
  Architecture:    Llama (decoder-only)
  Hidden size:     512
  Layers:          8
  Vocab size:      6251
  Device:          mps
  Max steps:       3
  Batch size:      1
  Learning rate:   0.0002
  Output dir:      /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/models/decoder-demo/checkpoints

Loading corpus from /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/data/decoder_corpus/smoke/train_512.txt...
  

  0%|          | 0/3 [00:00<?, ?it/s]/opt/anaconda3/envs/tfm/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 10.77it/s]


{'loss': '8.789', 'grad_norm': '15.74', 'learning_rate': '0', 'epoch': '0.001953'}
{'loss': '8.847', 'grad_norm': '21.47', 'learning_rate': '2e-05', 'epoch': '0.003906'}
{'loss': '8.858', 'grad_norm': '14.72', 'learning_rate': '4e-05', 'epoch': '0.005859'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 13.27it/s]


{'train_runtime': '0.8543', 'train_samples_per_second': '3.512', 'train_steps_per_second': '3.512', 'train_loss': '8.831', 'epoch': '0.005859'}

Model saved to /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/models/decoder-demo/checkpoints
returncode: 0


### Downsize to meet training cap

In [5]:
import subprocess

# Fresh training demo: 30 steps from scratch using HuggingFace Trainer.
# The loss will drop rapidly from ~8.7 (random baseline for ~6K vocab) demonstrating
# that the model is learning transaction patterns. The full pre-trained checkpoint
# is available for notebook 04.
DEMO_CHECKPOINT_DIR = PROJECT_ROOT / "models" / "decoder-demo" / "checkpoints"
DEMO_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    "python", str(TRAIN_SCRIPT),
    "--train_data", str(CORPUS_PATH),
    "--val_data", str(VAL_CORPUS_PATH),
    "--output_dir", str(DEMO_CHECKPOINT_DIR),
    "--max_steps", "30",
    ## downsize training
    "--seq_length", "4096",                  # 4096
    "--per_device_train_batch_size", "1",    # 16
    "--eval_steps", "1000",
    "--save_steps", "1000",
    "--logging_steps", "1",
]

print("Launching HuggingFace Trainer training (30 steps):")
print(" ".join(cmd))
print()

result = subprocess.run(cmd, capture_output=False, text=True, cwd=str(PROJECT_ROOT))

if result.returncode != 0:
    print(f"\nTraining exited with code {result.returncode}")
else:
    print("\nTraining complete!")
    print(f"\nThe full pre-trained model is available at:")
    print(f"  {PRETRAINED_MODEL}")

Launching HuggingFace Trainer training (30 steps):
python /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/scripts/train_decoder_hf.py --train_data /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/data/decoder_corpus/train_corpus.txt --val_data /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/data/decoder_corpus/val_corpus.txt --output_dir /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/models/decoder-demo/checkpoints --max_steps 30 --seq_length 4096 --per_device_train_batch_size 1 --eval_steps 1000 --save_steps 1000 --logging_steps 1


Financial Foundation Model — Decoder-Only Pretraining (HF)
  Architecture:    Llama (decoder-only)
  Hidden size:     512
  Layers:          8
  Vocab size:      6251
  Device:          mps
  Max steps:       30
  Batch size:      1
  Learning rate:   0.0002
  Output dir:      /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/models/decoder

  0%|          | 0/30 [00:00<?, ?it/s]/opt/anaconda3/envs/tfm/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 0/1218 [00:00<?, ?it/s]

{'loss': '8.907', 'grad_norm': '19', 'learning_rate': '0', 'epoch': '1.554e-05'}
{'loss': '8.833', 'grad_norm': '18.36', 'learning_rate': '2e-05', 'epoch': '3.109e-05'}
{'loss': '8.758', 'grad_norm': '20.62', 'learning_rate': '4e-05', 'epoch': '4.663e-05'}
{'loss': '8.818', 'grad_norm': '16.79', 'learning_rate': '6e-05', 'epoch': '6.217e-05'}
{'loss': '8.642', 'grad_norm': '20.81', 'learning_rate': '8e-05', 'epoch': '7.772e-05'}
{'loss': '8.686', 'grad_norm': '16.86', 'learning_rate': '0.0001', 'epoch': '9.326e-05'}
{'loss': '8.394', 'grad_norm': '12.78', 'learning_rate': '0.00012', 'epoch': '0.0001088'}
{'loss': '8.159', 'grad_norm': '10.54', 'learning_rate': '0.00014', 'epoch': '0.0001243'}
{'loss': '7.909', 'grad_norm': '7.598', 'learning_rate': '0.00016', 'epoch': '0.0001399'}
{'loss': '8.485', 'grad_norm': '18.73', 'learning_rate': '0.00018', 'epoch': '0.0001554'}
{'loss': '7.811', 'grad_norm': '5.871', 'learning_rate': '0.0002', 'epoch': '0.000171'}
{'loss': '7.786', 'grad_norm':


100%|█████████▉| 1217/1218 [1:50:39<00:05,  5.77s/it]
                                               5s/it]
100%|██████████| 1218/1218 [1:50:46<00:00,  5.75s/it]
                                                     
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.38it/s]


{'eval_loss': '6.906', 'eval_runtime': '6646', 'eval_samples_per_second': '1.465', 'eval_steps_per_second': '0.183', 'epoch': '0.0004663'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 12.11it/s]


{'train_runtime': '6661', 'train_samples_per_second': '0.005', 'train_steps_per_second': '0.005', 'train_loss': '7.563', 'epoch': '0.0004663'}

Model saved to /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/models/decoder-demo/checkpoints

Training complete!

The full pre-trained model is available at:
  /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/models/decoder-foundation-model


In [6]:
import os
import sys
import subprocess
from pathlib import Path

DEMO_CHECKPOINT_DIR = PROJECT_ROOT / "models" / "decoder-demo" / "checkpoints"
DEMO_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Small validation sample: avoids full 9,739-sequence eval after tiny demo train.
VAL_DEMO_PATH = CORPUS_DIR / "val_demo_256.txt"
with open(VAL_CORPUS_PATH, "r") as src, open(VAL_DEMO_PATH, "w") as dst:
  for i, line in enumerate(src):
      if i >= 256:
          break
      dst.write(line)

env = os.environ.copy()
env["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"

cmd = [
  sys.executable, str(TRAIN_SCRIPT),
  "--train_data", str(CORPUS_PATH),
  "--val_data", str(VAL_DEMO_PATH),
  "--output_dir", str(DEMO_CHECKPOINT_DIR),
  "--max_steps", "500",
  "--seq_length", "4096",
  "--per_device_train_batch_size", "1",
  "--eval_steps", "250",
  "--save_steps", "250",
  "--logging_steps", "10",
]

print("Launching Mac-safe HuggingFace Trainer run:")
print(" ".join(cmd))
print()

result = subprocess.run(cmd, capture_output=False, text=True, cwd=str(PROJECT_ROOT), env=env)

if result.returncode != 0:
  print(f"\nTraining exited with code {result.returncode}")
else:
  print("\nTraining complete!")
  print(f"Checkpoint saved to: {DEMO_CHECKPOINT_DIR}")

Launching Mac-safe HuggingFace Trainer run:
/opt/anaconda3/envs/tfm/bin/python3.12 /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/scripts/train_decoder_hf.py --train_data /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/data/decoder_corpus/train_corpus.txt --val_data /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/data/decoder_corpus/val_demo_256.txt --output_dir /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/models/decoder-demo/checkpoints --max_steps 500 --seq_length 4096 --per_device_train_batch_size 1 --eval_steps 250 --save_steps 250 --logging_steps 10


Financial Foundation Model — Decoder-Only Pretraining (HF)
  Architecture:    Llama (decoder-only)
  Hidden size:     512
  Layers:          8
  Vocab size:      6251
  Device:          mps
  Max steps:       500
  Batch size:      1
  Learning rate:   0.0002
  Output dir:      /Users/leon/Documents/03.LLM/_git-projects/transaction-fou

  0%|          | 0/500 [00:00<?, ?it/s]/opt/anaconda3/envs/tfm/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 0/32 [00:00<?, ?it/s]

{'loss': '8.547', 'grad_norm': '22.04', 'learning_rate': '0.00018', 'epoch': '0.0001554'}
{'loss': '7.327', 'grad_norm': '9.172', 'learning_rate': '0.0001963', 'epoch': '0.0003109'}
{'loss': '6.385', 'grad_norm': '3.722', 'learning_rate': '0.0001922', 'epoch': '0.0004663'}
{'loss': '5.854', 'grad_norm': '3.594', 'learning_rate': '0.0001882', 'epoch': '0.0006217'}
{'loss': '5.69', 'grad_norm': '4.065', 'learning_rate': '0.0001841', 'epoch': '0.0007772'}
{'loss': '5.679', 'grad_norm': '3.572', 'learning_rate': '0.00018', 'epoch': '0.0009326'}
{'loss': '5.441', 'grad_norm': '4.059', 'learning_rate': '0.0001759', 'epoch': '0.001088'}
{'loss': '5.314', 'grad_norm': '8.936', 'learning_rate': '0.0001718', 'epoch': '0.001243'}
{'loss': '5.264', 'grad_norm': '420.1', 'learning_rate': '0.0001678', 'epoch': '0.001399'}
{'loss': '5.149', 'grad_norm': '5.226', 'learning_rate': '0.0001637', 'epoch': '0.001554'}
{'loss': '4.938', 'grad_norm': '5.554', 'learning_rate': '0.0001596', 'epoch': '0.00171'}


 97%|█████████▋| 31/32 [02:43<00:05,  5.55s/it]
                                                 A
100%|██████████| 32/32 [02:54<00:00,  5.39s/it]
                                               
Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '2.556', 'eval_runtime': '174.5', 'eval_samples_per_second': '1.467', 'eval_steps_per_second': '0.183', 'epoch': '0.003886'}


...

{'loss': '2.813', 'grad_norm': '3.494', 'learning_rate': '9.837e-05', 'epoch': '0.004041'}
{'loss': '3.192', 'grad_norm': '7.578', 'learning_rate': '9.429e-05', 'epoch': '0.004197'}
{'loss': '2.987', 'grad_norm': '9.805', 'learning_rate': '9.02e-05', 'epoch': '0.004352'}
{'loss': '2.833', 'grad_norm': '0.9272', 'learning_rate': '8.612e-05', 'epoch': '0.004508'}
{'loss': '3', 'grad_norm': '3.353', 'learning_rate': '8.204e-05', 'epoch': '0.004663'}
{'loss': '2.902', 'grad_norm': '4.353', 'learning_rate': '7.796e-05', 'epoch': '0.004819'}
{'loss': '2.956', 'grad_norm': '3.628', 'learning_rate': '7.388e-05', 'epoch': '0.004974'}
{'loss': '2.945', 'grad_norm': '3.924', 'learning_rate': '6.98e-05', 'epoch': '0.005129'}
{'loss': '2.962', 'grad_norm': '3.763', 'learning_rate': '6.571e-05', 'epoch': '0.005285'}
{'loss': '2.778', 'grad_norm': '5.151', 'learning_rate': '6.163e-05', 'epoch': '0.00544'}
{'loss': '2.502', 'grad_norm': '4.137', 'learning_rate': '5.755e-05', 'epoch': '0.005596'}
{'los


 97%|█████████▋| 31/32 [02:42<00:05,  5.24s/it]
                                                 A
100%|██████████| 32/32 [02:52<00:00,  5.33s/it]
                                               
Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '2.242', 'eval_runtime': '172.8', 'eval_samples_per_second': '1.481', 'eval_steps_per_second': '0.185', 'epoch': '0.007772'}



Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 15.21it/s]


{'train_runtime': '562.6', 'train_samples_per_second': '0.889', 'train_steps_per_second': '0.889', 'train_loss': '3.774', 'epoch': '0.007772'}

Model saved to /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/models/decoder-demo/checkpoints

Training complete!
Checkpoint saved to: /Users/leon/Documents/03.LLM/_git-projects/transaction-foundation-model/models/decoder-demo/checkpoints


## Checkpoints

With `save_consolidated: true`, checkpoints are in HuggingFace format (`config.json` + `safetensors`), loadable via `AutoModelForCausalLM.from_pretrained()`. The pre-trained checkpoint at `models/decoder-foundation-model/` is ready for inference in notebook 04.

## Summary

We trained a \~29M parameter decoder foundation model on \~64K tokenized transaction sequences using NeMo AutoModel. The model learns to predict the next transaction token from context through self-supervised learning, without any fraud labels. The full pre-trained checkpoint (\~3,000 steps on 8x A100) is available for the next step.

Continue to [04_inference_embedding_extraction.ipynb](./04_inference_embedding_extraction.ipynb).